# Emotive TTS — Master Colab Pipeline

**COMP3065 Final Year Project**

This notebook runs all GPU-intensive tasks on Google Colab T4.
All code is developed locally — this notebook orchestrates compute only.

## Workflow
1. Mount Google Drive + clone repo
2. Install dependencies
3. Data preparation (S1)
4. Training: System A → B → C (S2-S4)
5. Inference: generate eval stimuli (S5)
6. Evaluation: prosody analysis + SER probe (S6)

**Runtime:** GPU → T4 (Runtime > Change runtime type)

## 0. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Project paths
DRIVE_PROJECT = '/content/drive/MyDrive/COMP3065_Project'
LOCAL_PROJECT = '/content/project'

import os
os.makedirs(DRIVE_PROJECT, exist_ok=True)

In [ ]:
# Clone or sync project repo
# Option A: Clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/COMP3065_Project.git {LOCAL_PROJECT}

# Option B: Copy from Drive
!cp -r {DRIVE_PROJECT}/code {LOCAL_PROJECT} 2>/dev/null || echo 'Upload project to Drive first'

# Or upload directly: use the file upload button in Colab
%cd {LOCAL_PROJECT}

In [ ]:
# Check GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q TTS==0.22.0 speechbrain==1.0.0 librosa==0.10.2
!pip install -q hydra-core==1.3.2 mlflow==2.12.1 gradio==4.44.0
!pip install -q pandas matplotlib seaborn scipy scikit-learn soundfile

# Install project in dev mode
!pip install -q -e .

# Verify
from src.data.utils import EMOTION_MAP
print('Project installed OK. Emotions:', EMOTION_MAP)

## 1. Data Preparation (S1)

In [ ]:
# Download EmoV-DB (if not already on Drive)
EMOVDB_DRIVE = f'{DRIVE_PROJECT}/data/raw/EmoV-DB'
EMOVDB_LOCAL = 'data/raw/EmoV-DB'

if os.path.exists(EMOVDB_DRIVE):
    print('EmoV-DB found on Drive, creating symlink...')
    os.makedirs('data/raw', exist_ok=True)
    !ln -sf {EMOVDB_DRIVE} {EMOVDB_LOCAL}
else:
    print('Download EmoV-DB:')
    print('  Option 1: Upload to Google Drive at', EMOVDB_DRIVE)
    print('  Option 2: Download directly (uncomment below)')
    # !pip install gdown
    # !gdown --id YOUR_GDRIVE_FILE_ID -O emovdb.zip
    # !unzip emovdb.zip -d data/raw/

In [ ]:
# Run data preparation pipeline
from src.data.prepare import prepare_dataset
import yaml

with open('configs/data.yaml', 'r') as f:
    data_cfg = yaml.safe_load(f)

# Override paths for Colab
data_cfg['dataset']['raw_dir'] = EMOVDB_LOCAL

summary = prepare_dataset(data_cfg)
print('\nDataset summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

In [ ]:
# Run Data QA
from src.data.qa import generate_qa_report

generate_qa_report(
    manifest_path='data/manifests/full_manifest.csv',
    output_dir='figures/data_qa',
    report_path='docs/data_qa_report.md',
)
print('QA report generated: docs/data_qa_report.md')

## 2. Training: System A (S2)

In [ ]:
# Train System A: domain adaptation, no emotion labels
from src.training.train import train
import yaml

with open('configs/train_a.yaml', 'r') as f:
    train_a_cfg = yaml.safe_load(f)

# Colab overrides
train_a_cfg['training']['use_cuda'] = True
train_a_cfg['training']['fp16'] = True

print('Starting System A training...')
result_a = train(train_a_cfg)
print('System A training complete:', result_a)

In [ ]:
# Save System A checkpoint to Drive
!cp -r checkpoints/system_a {DRIVE_PROJECT}/checkpoints/
print('System A checkpoint saved to Drive')

## 3. Training: System B (S3)

In [ ]:
# Train System B: System A + emotion embedding
with open('configs/train_b.yaml', 'r') as f:
    train_b_cfg = yaml.safe_load(f)

train_b_cfg['training']['use_cuda'] = True
train_b_cfg['training']['fp16'] = True

print('Starting System B training...')
result_b = train(train_b_cfg)
print('System B training complete:', result_b)

In [ ]:
!cp -r checkpoints/system_b {DRIVE_PROJECT}/checkpoints/
print('System B checkpoint saved to Drive')

## 4. Training: System C (S4)

In [ ]:
# Train System C: System B + prosody auxiliary heads
with open('configs/train_c.yaml', 'r') as f:
    train_c_cfg = yaml.safe_load(f)

train_c_cfg['training']['use_cuda'] = True
train_c_cfg['training']['fp16'] = True

print('Starting System C training...')
result_c = train(train_c_cfg)
print('System C training complete:', result_c)

In [ ]:
!cp -r checkpoints/system_c {DRIVE_PROJECT}/checkpoints/
print('System C checkpoint saved to Drive')

## 5. Inference (S5)

Generate evaluation stimuli: 16 canary texts × 4 emotions × 4 systems

In [ ]:
from src.inference.run import run_inference
import yaml

with open('configs/infer.yaml', 'r') as f:
    infer_cfg = yaml.safe_load(f)

infer_cfg['inference']['use_cuda'] = True

results = run_inference(infer_cfg)
print('Inference complete:', results)

In [ ]:
# Save eval stimuli to Drive
!cp -r data/processed/eval_stimuli {DRIVE_PROJECT}/data/processed/
print('Eval stimuli saved to Drive')

## 6. Evaluation (S6)

In [ ]:
# 6a. Prosody analysis (PRIMARY metric)
from src.evaluation.prosody import run_prosody_evaluation
import yaml

with open('configs/eval.yaml', 'r') as f:
    eval_cfg = yaml.safe_load(f)

prosody_results = run_prosody_evaluation(eval_cfg)
print('Prosody evaluation complete')

# Show aggregate stats
if 'agg_stats' in prosody_results:
    display(prosody_results['agg_stats'])

In [ ]:
# 6b. SER probe (AUXILIARY metric — label mismatch caveat applies)
from src.evaluation.ser_probe import run_ser_evaluation

ser_results = run_ser_evaluation(eval_cfg)
print(f"\nSER proxy agreement: {ser_results['ser_proxy_agreement']:.2%}")
print(f"Note: This is an AUXILIARY metric only.")
print(f"Excluded {ser_results['n_excluded']} samples with unmapped emotions (disgust).")

In [ ]:
# 6c. Generate all plots
from src.evaluation.plots import generate_all_plots

generate_all_plots(eval_cfg)
print('All plots saved to figures/')

# Display key plots inline
from IPython.display import Image, display
for plot_name in ['f0_by_system_emotion.png', 'prosody_comparison_grid.png']:
    plot_path = f'figures/{plot_name}'
    if os.path.exists(plot_path):
        display(Image(filename=plot_path, width=800))

In [ ]:
# 6d. Generate listening test stimulus pack
from src.evaluation.listening_test import create_stimulus_pack

stim_summary = create_stimulus_pack(
    manifest_path='data/processed/eval_stimuli/eval_manifest.csv',
    output_dir='outputs/listening_test',
)
print('Stimulus pack:', stim_summary)

In [ ]:
# Save all outputs to Drive
!cp -r tables {DRIVE_PROJECT}/
!cp -r figures {DRIVE_PROJECT}/
!cp -r outputs {DRIVE_PROJECT}/
!cp -r docs {DRIVE_PROJECT}/
print('All outputs saved to Drive')

## 7. Summary

| Step | Status |
|------|--------|
| Data prep | ✅ |
| System A training | ✅ |
| System B training | ✅ |
| System C training | ✅ |
| Inference | ✅ |
| Prosody eval | ✅ |
| SER probe | ✅ |
| Plots | ✅ |
| Listening test | ✅ |

All checkpoints and outputs are saved to Google Drive.